# Notebook 00: Set up, simulated sample data, freeze the gene sets

**Goal.** Create a realistic **simulated** dataset (2 cohorts, 19 mice, about 8,500 cells) with a known answer, so you can run the entire pipeline before touching real data; then freeze the gene sets.

Why simulate? Real downloads are large and their file formats vary. The simulated data have the same structure as real data, so if every notebook runs here, any later problem is about your files, not your installation. The simulation contains: (1) coupling that falls with age, (2) more disease-associated (DAM) cells with age, (3) a few contaminating non-microglia cells, low-quality cells and doublets, (4) six 'interface' genes that carry the coupling.

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** **synth.make_sample_data()** writes two cohorts (SIM_A: 11 mice at 3, 12, 24 months; SIM_B: 8 mice at 4 and 22 months), a sample sheet, human bulk data, a gene-level GWAS table, a CCCP table and two colocalisation loci into data/. **set_profile** records that you are in sample mode.

*You should see:* profile set to sample - restart this notebook's kernel before continuing

In [ ]:
from mlc import synth
synth.make_sample_data()          # about 10 seconds
set_profile("sample")

**Cell 3.** The **sample sheet** has one row per biological sample (mouse). This is the most important table in the project: every statistic later uses the mouse, not the cell, as the unit.

*You should see:* Prints 6 rows of the sheet and a count of mice per cohort and age (4+3+4 for SIM_A, 4+4 for SIM_B).

In [ ]:
sheet = pd.read_csv(DATA / "sample_sheet_sample.csv")
print(sheet.head(6).to_string(index=False))
print(sheet.groupby(["cohort", "age_months"]).size())

**Cell 4.** Loads one simulated cohort. A **Dataset** holds a sparse count matrix (cells x genes), a table about cells (obs) and a table about genes (var). **Note:** columns that start with *true_* exist only in simulated data (they store the answer); real data never has them and the analysis never uses them.

*You should see:* 4950 cells x 1500 genes; median UMI per cell about 1,600; 450 cells per sample.

In [ ]:
from mlc.dataset import Dataset
ds = Dataset.load(PROC / "SIM_A_raw")
print(ds)
print("median UMI per cell:", int(np.median(np.asarray(ds.X.sum(axis=1)).ravel())))
print("cells per sample:"); print(ds.obs.sample_id.value_counts().sort_index().head(4))

**Cell 5.** **Freezing.** Writes every gene list into one file and computes its SHA-256 fingerprint (any change to any gene changes the fingerprint). Paste this fingerprint into your OSF pre-registration. It proves the gene sets were fixed *before* you looked at results.

*You should see:* A 64-character hexadecimal string.

In [ ]:
import hashlib, json, datetime
sets = {k: getattr(G, k) for k in dir(G) if k.isupper()}
blob = json.dumps(sets, sort_keys=True).encode()
digest = hashlib.sha256(blob).hexdigest()
(RES / "genesets_frozen.json").write_text(json.dumps(sets, indent=1))
(RES / "genesets_frozen.sha256").write_text(f"{digest}  {datetime.date.today()}\n")
print(digest)